# 03 — Gold enrichment (CSV inbox → tabela UC → join)

Este notebook:
- Lê o CSV de enrichment do **container gold** (inbox)
- Publica como tabela `kabum_project.gold.enrichment_raw`
- Faz join com `kabum_project.silver.products_clean`
- Grava `kabum_project.gold.notebooks_features_enriched`

In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
spark.sql("USE CATALOG projeto_data_engineering")
spark.sql("USE SCHEMA kabum_project")

STORAGE_ACCOUNT = "storagekabumdata"

# CSV do enrichment (coloque o arquivo aqui no Azure Storage)
GOLD_ENRICH_INBOX_DIR = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/inbox/kabum/gold_enrichment_v1/"

ENRICH_RAW_TABLE    = "projeto_data_engineering.kabum_project.enrichment_raw"
SILVER_TABLE        = "projeto_data_engineering.kabum_project.products_clean"
GOLD_ENRICHED_TABLE = "projeto_data_engineering.kabum_project.notebooks_features_enriched"

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {ENRICH_RAW_TABLE}")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# 1) Ingest CSV enrichment -> tabela (RAW)
#    - Lemos tudo como STRING para evitar drift de schema (CSV varia muito)
#    - A conversão para tipos numéricos acontece no join (etapa 2)
# =========================================================
df_enrich_raw = (
    spark.read
      .option("header", True)
      .option("inferSchema", False)   # <- força string
      .csv(GOLD_ENRICH_INBOX_DIR)
)

# Normaliza nomes (opcional) e garante colunas esperadas como string
for c in ["refresh_rate_hz","brightness_nits","price","old_price","screen_resolution","panel_type","panel_finish","scrape_error","product_url"]:
    if c in df_enrich_raw.columns:
        df_enrich_raw = df_enrich_raw.withColumn(c, F.col(c).cast("string"))

(df_enrich_raw.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(ENRICH_RAW_TABLE)
)

print("OK ->", ENRICH_RAW_TABLE, "rows:", df_enrich_raw.count())


OK -> projeto_data_engineering.kabum_project.enrichment_raw rows: 79


In [0]:
from pyspark.sql import functions as F

# =========================================================
# 2) Join Silver + Enrichment (robusto)
#    Objetivo:
#    - manter RAW com strings (enrichment_raw)
#    - converter colunas _scraped para tipos estáveis ANTES do join/write (evita DELTA_FAILED_TO_MERGE_FIELDS)
#    - sobrescrever somente as partições do(s) ingestion_date presentes (replaceWhere)
# =========================================================

df_silver = spark.read.table(SILVER_TABLE)
df_enrich = spark.read.table(ENRICH_RAW_TABLE)

# ---------------------------------------------------------
# 2.1) Garantir ingestion_date (e tipo date) em ambos
# ---------------------------------------------------------
def ensure_ingestion_date(df):
    if "ingestion_date" not in df.columns:
        return df.withColumn("ingestion_date", F.current_date())
    return df.withColumn("ingestion_date", F.to_date(F.col("ingestion_date").cast("string")))

df_silver = ensure_ingestion_date(df_silver)
df_enrich = ensure_ingestion_date(df_enrich)

# ---------------------------------------------------------
# 2.2) Define chave de join (prioridade: product_key -> product_url -> product_id)
# ---------------------------------------------------------
join_key = None
for k in ["product_key", "product_url", "product_id"]:
    if k in df_silver.columns and k in df_enrich.columns:
        join_key = k
        break

if join_key is None:
    raise Exception(
        "Não encontrei uma chave comum entre Silver e Enrichment. "
        "Tente garantir que ambos tenham 'product_key' ou 'product_url'."
    )

print("Join key:", join_key)

# ---------------------------------------------------------
# 2.3) Evita colisão de colunas: renomeia as do enrichment com sufixo _scraped (exceto a chave)
# ---------------------------------------------------------
enrich_cols = [c for c in df_enrich.columns if c != join_key]

df_enrich_ren = df_enrich.select(
    F.col(join_key),
    *[F.col(c).alias(f"{c}_scraped") for c in enrich_cols]
)

# ---------------------------------------------------------
# 2.4) Normalizações de tipos do enrichment (_scraped)
#      - Seu GOLD já tem refresh_rate_hz_scraped e brightness_nits_scraped como DOUBLE
#      - Então forçamos DOUBLE aqui (extraindo números de strings tipo "120Hz", "300 nits", etc.)
# ---------------------------------------------------------
def to_double_num(colname: str):
    s = F.col(colname).cast("string")
    s = F.regexp_replace(s, r"[^0-9,\.]", "")  # remove tudo que não for número/vírgula/ponto
    s = F.regexp_replace(s, r",", ".")          # vírgula decimal -> ponto
    return F.when(F.length(s) == 0, F.lit(None)).otherwise(s.cast("double"))

for c in ["refresh_rate_hz_scraped", "brightness_nits_scraped", "price_scraped", "old_price_scraped"]:
    if c in df_enrich_ren.columns:
        df_enrich_ren = df_enrich_ren.withColumn(c, to_double_num(c))

# strings garantidas
for c in ["screen_resolution_scraped","panel_type_scraped","panel_finish_scraped","scrape_error_scraped","product_url_scraped"]:
    if c in df_enrich_ren.columns:
        df_enrich_ren = df_enrich_ren.withColumn(c, F.col(c).cast("string"))

# boolean garantido (às vezes vem como string)
if "scrape_ok_scraped" in df_enrich_ren.columns:
    df_enrich_ren = df_enrich_ren.withColumn(
        "scrape_ok_scraped",
        F.when(F.col("scrape_ok_scraped").cast("string").isin("true", "True", "1"), F.lit(True))
         .when(F.col("scrape_ok_scraped").cast("string").isin("false", "False", "0"), F.lit(False))
         .otherwise(F.col("scrape_ok_scraped").cast("boolean"))
    )

# ---------------------------------------------------------
# 2.5) Join
# ---------------------------------------------------------
df_join = df_silver.join(df_enrich_ren, on=join_key, how="left")

# ---------------------------------------------------------
# 2.6) Coalesce: se existir a mesma coluna sem sufixo no Silver, preenche com scraped
# ---------------------------------------------------------
for c in enrich_cols:
    sc = c
    ec = f"{c}_scraped"
    if sc in df_join.columns and ec in df_join.columns:
        df_join = df_join.withColumn(sc, F.coalesce(F.col(sc), F.col(ec)))

df_join = df_join.withColumn(
    "enriched_from_scrape",
    F.col(enrich_cols[0] + "_scraped").isNotNull() if enrich_cols else F.lit(False)
)

# =========================================================
# 3) Write Gold Enriched
#    - 1ª vez: cria tabela particionada (ingestion_date, search_term se existir)
#    - demais execuções: overwrite SOMENTE a(s) ingestion_date presente(s) (replaceWhere)
# =========================================================
table_exists = spark.catalog.tableExists(GOLD_ENRICHED_TABLE)

part_cols = []
if "ingestion_date" in df_join.columns:
    part_cols.append("ingestion_date")
if "search_term" in df_join.columns:
    part_cols.append("search_term")

base_writer = (
    df_join.write
      .format("delta")
      .option("mergeSchema", "true")
      .option("overwriteSchema", "true")
)

# datas presentes no lote
ing_dates = []
if "ingestion_date" in df_join.columns:
    ing_dates = [r["ingestion_date"] for r in df_join.select("ingestion_date").distinct().collect() if r["ingestion_date"] is not None]

if not table_exists:
    writer = base_writer.mode("overwrite")
    if part_cols:
        writer = writer.partitionBy(*part_cols)
    writer.saveAsTable(GOLD_ENRICHED_TABLE)
    print(f"✅ Criada Gold: {GOLD_ENRICHED_TABLE} | rows={df_join.count()}")
else:
    # Se não houver ingestion_date (não deveria), faz overwrite total
    if not ing_dates:
        (base_writer.mode("overwrite")
          .saveAsTable(GOLD_ENRICHED_TABLE))
        print(f"✅ Overwrite total (sem ingestion_date) em: {GOLD_ENRICHED_TABLE} | rows={df_join.count()}")
    else:
        for d in ing_dates:
            d_str = str(d)  # yyyy-mm-dd
            df_part = df_join.filter(F.col("ingestion_date") == F.lit(d))
            (df_part.write.format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .option("overwriteSchema", "true")
              .option("replaceWhere", f"ingestion_date = '{d_str}'")
              .saveAsTable(GOLD_ENRICHED_TABLE))
            print(f"✅ Overwrite por ingestion_date={d_str} em: {GOLD_ENRICHED_TABLE} | rows={df_part.count()}")


Join key: product_key


In [0]:
display(spark.read.table(GOLD_ENRICHED_TABLE).limit(30))

product_key,ingestion_date,marketplace,search_term,kabum_code,product_name,brand_id,brand,brand_img,category,available,currency,price,old_price,discount_pct,rating,reviews_count,max_installment,product_url,image_url,scraped_at,price_raw,old_price_raw,discount_pct_raw,discount_pct_site,discount_pct_gap,product_url_scraped,refresh_rate_hz_scraped,screen_resolution_scraped,panel_type_scraped,panel_finish_scraped,brightness_nits_scraped,scrape_ok_scraped,scrape_error_scraped,enriched_from_scrape
0005fe5fae3c3d002f547418ceedfe52a766c420c9ece022d716a04f43e0798d,null,kabum,notebook,883577,"Notebook Lenovo Ideapad Slim 3 15irh10 Intel Core i5-13420h 16gb 512gb SSD WINDOWS 11 15.3"" - 83ns0001br Luna Grey",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,3695.12,null,0,5.0,3,"10x de R$ 419,90",https://www.kabum.com.br/produto/883577/notebook-lenovo-ideapad-slim-3-15irh10-intel-core-i5-13420h-16gb-512gb-ssd-windows-11-15-3-83ns0001br-luna-grey,https://images.kabum.com.br/produtos/fotos/sync_mirakl/883577/xlarge/Notebook-Lenovo-Ideapad-Slim-3-15irh10-Intel-Core-i5-13420h-16gb-512gb-SSD-WINDOWS-11-15-3-83ns0001br-Luna-Grey_1770929879.jpg,2026-03-04T21:24:48.619Z,null,null,12.0,12.0,12.0,https://www.kabum.com.br/produto/883577/notebook-lenovo-ideapad-slim-3-15irh10-intel-core-i5-13420h-16gb-512gb-ssd-windows-11-15-3-83ns0001br-luna-grey,60.0,1920x1200,IPS,MATTE,300.0,true,null,true
0461b3e06c104ca6956d220d00e9ae48fb5a8d6502573d38e450799e45eb03b7,null,kabum,notebook,627471,Notebook Asus Vivobook 16 X1605va Intel Core I7 1355u 16gb Ram 512gb SSD WINDOWS 11 Home Tela 16” Ips Fhd Silver - Mb763w,1,ASUS,https://images4.kabum.com.br/produtos/fabricantes/logo-asus.jpg,Computadores/Notebooks/Notebook Asus,true,BRL,4304.81,4836.87,11,5.0,1,"10x de R$ 435,31",https://www.kabum.com.br/produto/627471/notebook-asus-vivobook-16-x1605va-intel-core-i7-1355u-16gb-ram-512gb-ssd-windows-11-home-tela-16-ips-fhd-silver-mb763w,https://images.kabum.com.br/produtos/fotos/sync_mirakl/627471/xlarge/Notebook-Asus-Vivobook-16-X1605va-Intel-Core-I7-1355u-16gb-Ram-512gb-SSD-WINDOWS-11-Home-Tela-16-Ips-Fhd-Silver-Mb763w_1765585717.jpg,2026-03-04T21:24:48.619Z,null,null,11.0,11.0,0.0,https://www.kabum.com.br/produto/627471/notebook-asus-vivobook-16-x1605va-intel-core-i7-1355u-16gb-ram-512gb-ssd-windows-11-home-tela-16-ips-fhd-silver-mb763w,null,1920x1200,IPS,null,null,true,null,true
18c57d400cb2cf0d50d171179055b0cbccbda123bbd16e26be37018064510784,null,kabum,notebook,879301,"Notebook Lenovo Loq-e 15iax9e Intel Core i7-12650hx 16gb 512gb SSD RTX 4050 Linux 15.6"" - 83mes00200 Luna Grey",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,7202.6,null,0,5.0,2,"10x de R$ 827,88",https://www.kabum.com.br/produto/879301/notebook-lenovo-loq-e-15iax9e-intel-core-i7-12650hx-16gb-512gb-ssd-rtx-4050-linux-15-6-83mes00200-luna-grey,https://images.kabum.com.br/produtos/fotos/sync_mirakl/879301/xlarge/Notebook-Lenovo-Loq-e-15iax9e-Intel-Core-i7-12650hx-16gb-512gb-SSD-RTX-4050-Linux-15-6-83mes00200-Luna-Grey_1765465529.jpg,2026-03-04T21:24:48.619Z,null,null,13.0,13.0,13.0,https://www.kabum.com.br/produto/879301/notebook-lenovo-loq-e-15iax9e-intel-core-i7-12650hx-16gb-512gb-ssd-rtx-4050-linux-15-6-83mes00200-luna-grey,144.0,1920x1080,IPS,MATTE,300.0,true,null,true
2c67f0724aea643dd2eb457186d9d7c64d7b513210f393ddf877ad5ed3621866,null,kabum,notebook,728904,"Notebook Lenovo LOQ-E Core i5-12450HX, RTX 3050, 16GB, 512GB SSD, 15.6"" FHD, IPS, 144Hz, Win 11, Luna Cinza- 83ME0007BR",205,Lenovo,https://images4.kabum.com.br/produtos/fabricantes/logo-lenovo.jpg,Computadores/Notebooks/Notebook Lenovo,true,BRL,5947.53,null,0,5.0,73,"10x de R$ 660,83",https://www.kabum.com.br/produto/728904/notebook-lenovo-loq-e-core-i5-12450hx-rtx-3050-16gb-512gb-ssd-15-6-fhd-ips-144hz-win-11-luna-cinza-83me0007br,https://images.kabum.com.br/produtos/fotos/728904/noteboo